In [1]:
import pybullet as p
import numpy as np
import sys
from pathlib import Path
import os

In [2]:
datalabs_path = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'simulation' / 'cloned').exists()), Path.cwd().parent)
sys.path.insert(0, str(datalabs_path))

# Import the class
from simulation.cloned.sim_class import Simulation

# Change to the directory where sim_class.py expects to be run from
original_dir = Path.cwd()
sim_dir = datalabs_path / "simulation" / "cloned"
os.chdir(sim_dir)

In [4]:
# Create simulation on first run, reset on subsequent runs
if 'test_sim' not in globals():
    print("Creating new simulation...")
    test_sim = Simulation(num_agents=1, render=True)
else:
    print("Resetting robot...")
    test_sim.reset(num_agents=1)

# Discover actual working envelope by driving to limits
print(f"\n{'='*80}")
print("WORKSPACE BOUNDARY DISCOVERY")
print(f"{'='*80}\n")

robot_id = test_sim.robotIds[0]
test_sim.set_start_position(0, 0, 0.2)

# Find Z max FIRST (go up from initial position)
print("Finding Z MAX (moving up)...")
for step in range(200):
    test_sim.run([[0, 0, 0.5, 0]], num_steps=1)
    if step % 20 == 0:
        states = test_sim.get_states()
        pos = states[f'robotId_{robot_id}']['pipette_position']
        print(f"  Step {step:3d} | Position: X={pos[0]:7.4f}m  Y={pos[1]:7.4f}m  Z={pos[2]:7.4f}m")
states = test_sim.get_states()
z_max = states[f'robotId_{robot_id}']['pipette_position'][2]
print(f"  Z MAX found: {z_max:.4f}m\n")

# Find Z min (go down)
print("Finding Z MIN (moving down)...")
for step in range(400):
    test_sim.run([[0, 0, -0.5, 0]], num_steps=1)
    if step % 40 == 0:
        states = test_sim.get_states()
        pos = states[f'robotId_{robot_id}']['pipette_position']
        print(f"  Step {step:3d} | Position: X={pos[0]:7.4f}m  Y={pos[1]:7.4f}m  Z={pos[2]:7.4f}m")
states = test_sim.get_states()
z_min = states[f'robotId_{robot_id}']['pipette_position'][2]
print(f"  Z MIN found: {z_min:.4f}m\n")

# Return to safe Z and find X max
print("Finding X MAX (moving right)...")
for step in range(200):
    test_sim.run([[0, 0, 0.5, 0]], num_steps=1)
for step in range(200):
    test_sim.run([[0.5, 0, 0, 0]], num_steps=1)
    if step % 20 == 0:
        states = test_sim.get_states()
        pos = states[f'robotId_{robot_id}']['pipette_position']
        print(f"  Step {step:3d} | Position: X={pos[0]:7.4f}m  Y={pos[1]:7.4f}m  Z={pos[2]:7.4f}m")
states = test_sim.get_states()
x_max = states[f'robotId_{robot_id}']['pipette_position'][0]
print(f"  X MAX found: {x_max:.4f}m\n")

# Find X min
print("Finding X MIN (moving left)...")
for step in range(400):
    test_sim.run([[-0.5, 0, 0, 0]], num_steps=1)
    if step % 40 == 0:
        states = test_sim.get_states()
        pos = states[f'robotId_{robot_id}']['pipette_position']
        print(f"  Step {step:3d} | Position: X={pos[0]:7.4f}m  Y={pos[1]:7.4f}m  Z={pos[2]:7.4f}m")
states = test_sim.get_states()
x_min = states[f'robotId_{robot_id}']['pipette_position'][0]
print(f"  X MIN found: {x_min:.4f}m\n")

# Return to center and find Y max
print("Finding Y MAX (moving forward)...")
for step in range(200):
    test_sim.run([[0.5, 0, 0, 0]], num_steps=1)
for step in range(200):
    test_sim.run([[0, 0.5, 0, 0]], num_steps=1)
    if step % 20 == 0:
        states = test_sim.get_states()
        pos = states[f'robotId_{robot_id}']['pipette_position']
        print(f"  Step {step:3d} | Position: X={pos[0]:7.4f}m  Y={pos[1]:7.4f}m  Z={pos[2]:7.4f}m")
states = test_sim.get_states()
y_max = states[f'robotId_{robot_id}']['pipette_position'][1]
print(f"  Y MAX found: {y_max:.4f}m\n")

# Find Y min
print("Finding Y MIN (moving backward)...")
for step in range(400):
    test_sim.run([[0, -0.5, 0, 0]], num_steps=1)
    if step % 40 == 0:
        states = test_sim.get_states()
        pos = states[f'robotId_{robot_id}']['pipette_position']
        print(f"  Step {step:3d} | Position: X={pos[0]:7.4f}m  Y={pos[1]:7.4f}m  Z={pos[2]:7.4f}m")
states = test_sim.get_states()
y_min = states[f'robotId_{robot_id}']['pipette_position'][1]
print(f"  Y MIN found: {y_min:.4f}m\n")

# Now move to all 8 corners
print(f"{'='*80}")
print("VISITING ALL 8 CORNERS")
print(f"{'='*80}\n")

corners = [
    (x_min, y_min, z_min),
    (x_min, y_min, z_max),
    (x_min, y_max, z_min),
    (x_min, y_max, z_max),
    (x_max, y_min, z_min),
    (x_max, y_min, z_max),
    (x_max, y_max, z_min),
    (x_max, y_max, z_max),
]

for i, (target_x, target_y, target_z) in enumerate(corners, 1):
    print(f"\nCorner {i}/8: Target = ({target_x:.4f}, {target_y:.4f}, {target_z:.4f})")
    
    # Move to target position
    max_attempts = 500
    for attempt in range(max_attempts):
        states = test_sim.get_states()
        current_pos = states[f'robotId_{robot_id}']['pipette_position']
        
        # Calculate direction to target
        dx = target_x - current_pos[0]
        dy = target_y - current_pos[1]
        dz = target_z - current_pos[2]
        
        # Print position every 50 steps
        if attempt % 50 == 0:
            distance = np.sqrt(dx**2 + dy**2 + dz**2)
            print(f"  Step {attempt:3d} | Position: X={current_pos[0]:7.4f}m  Y={current_pos[1]:7.4f}m  Z={current_pos[2]:7.4f}m | Distance: {distance*1000:6.2f}mm")
        
        # Check if close enough
        if abs(dx) < 0.01 and abs(dy) < 0.01 and abs(dz) < 0.01:
            print(f"  ✓ Reached corner {i}")
            break
        
        # Normalize and apply movement
        action = [
            np.sign(dx) * 0.5,
            np.sign(dy) * 0.5,
            np.sign(dz) * 0.5,
            0
        ]
        test_sim.run([action], num_steps=1)

print(f"\n{'='*80}")
print("WORKSPACE DISCOVERY COMPLETE")
print(f"{'='*80}")
print(f"X bounds: ({x_min:.4f}, {x_max:.4f})m")
print(f"Y bounds: ({y_min:.4f}, {y_max:.4f})m")
print(f"Z bounds: ({z_min:.4f}, {z_max:.4f})m")
print(f"{'='*80}\n")
print("Simulation window remains open. Rerun this cell to repeat the process.")

Resetting robot...

WORKSPACE BOUNDARY DISCOVERY

Finding Z MAX (moving up)...
  Step   0 | Position: X= 0.1460m  Y= 0.1790m  Z= 0.2010m
  Step  20 | Position: X= 0.1460m  Y= 0.1790m  Z= 0.2425m
  Step  40 | Position: X= 0.1460m  Y= 0.1790m  Z= 0.2841m
  Step  60 | Position: X= 0.1460m  Y= 0.1790m  Z= 0.2895m
  Step  80 | Position: X= 0.1460m  Y= 0.1790m  Z= 0.2895m
  Step 100 | Position: X= 0.1460m  Y= 0.1790m  Z= 0.2895m
  Step 120 | Position: X= 0.1460m  Y= 0.1790m  Z= 0.2895m
  Step 140 | Position: X= 0.1460m  Y= 0.1790m  Z= 0.2895m
  Step 160 | Position: X= 0.1460m  Y= 0.1790m  Z= 0.2895m
  Step 180 | Position: X= 0.1460m  Y= 0.1790m  Z= 0.2895m
  Z MAX found: 0.2895m

Finding Z MIN (moving down)...
  Step   0 | Position: X= 0.1460m  Y= 0.1790m  Z= 0.2882m
  Step  40 | Position: X= 0.1460m  Y= 0.1791m  Z= 0.2049m
  Step  80 | Position: X= 0.1460m  Y= 0.1791m  Z= 0.1695m
  Step 120 | Position: X= 0.1460m  Y= 0.1791m  Z= 0.1695m
  Step 160 | Position: X= 0.1460m  Y= 0.1791m  Z= 0.16